# LeetCode #1463: Cherry Pickup II

https://leetcode.com/problems/cherry-pickup-ii/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(3^{2 \cdot rows})$ | $O(rows)$ |
| **Optimal: 3D DP ★** | $O(rows \cdot cols^2 \cdot 9)$ | $O(cols^2)$ |

---

## Understanding the Methods

### Brute Force
Recursively enumerate all paths for both robots simultaneously (each has 3 choices per row) without memoization. Exponential — infeasible beyond tiny grids.

### Optimal: 3D DP ★
`dp[r][c1][c2]` = maximum cherries collectible when robot 1 is at column `c1` and robot 2 is at column `c2` on row `r`. Move both robots simultaneously: each can shift ±1 or stay. If both land on the same cell, count cherries once. Propagate row by row and discard previous rows (rolling 2D array), giving $O(cols^2)$ space.

**Why this is better than Brute Force:** The DP has $O(rows \times cols^2)$ states with 9 transitions each — polynomial versus exponential.

**Constraints:**
* `rows == grid.length`, `cols == grid[i].length`
* $2 \leq rows, cols \leq 70$
* $0 \leq$ `grid[i][j]` $\leq 100$

## Solutions

### C#

In [ ]:
public class Solution {
    public int CherryPickup(int[][] grid) {
        int rows = grid.Length, cols = grid[0].Length;
        // dp[c1][c2] = max cherries with robot1 at col c1, robot2 at col c2 (current row)
        var dp = new int[cols, cols];

        // Initialise for row 0: robot1 starts at col 0, robot2 at col cols-1
        for (int c1 = 0; c1 < cols; c1++)
            for (int c2 = 0; c2 < cols; c2++)
                dp[c1, c2] = int.MinValue;
        dp[0, cols - 1] = grid[0][0] + (cols > 1 ? grid[0][cols - 1] : 0);

        for (int r = 1; r < rows; r++) {
            var ndp = new int[cols, cols];
            for (int c1 = 0; c1 < cols; c1++)
                for (int c2 = 0; c2 < cols; c2++)
                    ndp[c1, c2] = int.MinValue;

            for (int c1 = 0; c1 < cols; c1++) {
                for (int c2 = 0; c2 < cols; c2++) {
                    if (dp[c1, c2] == int.MinValue) continue;
                    // Try all 9 combinations of moves for both robots
                    for (int d1 = -1; d1 <= 1; d1++) {
                        int nc1 = c1 + d1;
                        if (nc1 < 0 || nc1 >= cols) continue;
                        for (int d2 = -1; d2 <= 1; d2++) {
                            int nc2 = c2 + d2;
                            if (nc2 < 0 || nc2 >= cols) continue;
                            // Collect cherries; don't double-count if robots overlap
                            int cherries = grid[r][nc1] + (nc1 != nc2 ? grid[r][nc2] : 0);
                            int val = dp[c1, c2] + cherries;
                            if (val > ndp[nc1, nc2]) ndp[nc1, nc2] = val;
                        }
                    }
                }
            }
            dp = ndp;
        }

        int ans = 0;
        foreach (int v in dp) if (v > ans) ans = v;
        return ans;
    }
}

### Python

In [ ]:
from typing import List

class Solution:
    def cherry_pickup(self, grid: List[List[int]]) -> int:
        rows, cols = len(grid), len(grid[0])
        NEG_INF = float('-inf')
        # dp[c1][c2] = max cherries with robot1 at col c1, robot2 at col c2
        dp = [[NEG_INF] * cols for _ in range(cols)]
        dp[0][cols - 1] = grid[0][0] + (grid[0][cols - 1] if cols > 1 else 0)

        for r in range(1, rows):
            ndp = [[NEG_INF] * cols for _ in range(cols)]
            for c1 in range(cols):
                for c2 in range(cols):
                    if dp[c1][c2] == NEG_INF:
                        continue
                    # Try all 9 combinations of moves for both robots
                    for d1 in (-1, 0, 1):
                        nc1 = c1 + d1
                        if not 0 <= nc1 < cols:
                            continue
                        for d2 in (-1, 0, 1):
                            nc2 = c2 + d2
                            if not 0 <= nc2 < cols:
                                continue
                            # Collect cherries; don't double-count if robots overlap
                            cherries = grid[r][nc1] + (grid[r][nc2] if nc1 != nc2 else 0)
                            val = dp[c1][c2] + cherries
                            if val > ndp[nc1][nc2]:
                                ndp[nc1][nc2] = val
            dp = ndp

        return max(v for row in dp for v in row if v != NEG_INF)

### Go

In [ ]:
func cherryPickup(grid [][]int) int {
    rows, cols := len(grid), len(grid[0])
    const NEG = -(1 << 30)
    // dp[c1][c2] = max cherries with robot1 at col c1, robot2 at col c2
    dp := make([][]int, cols)
    for i := range dp { dp[i] = make([]int, cols); for j := range dp[i] { dp[i][j] = NEG } }
    first := grid[0][0]
    if cols > 1 { first += grid[0][cols-1] }
    dp[0][cols-1] = first

    for r := 1; r < rows; r++ {
        ndp := make([][]int, cols)
        for i := range ndp { ndp[i] = make([]int, cols); for j := range ndp[i] { ndp[i][j] = NEG } }
        for c1 := 0; c1 < cols; c1++ {
            for c2 := 0; c2 < cols; c2++ {
                if dp[c1][c2] == NEG { continue }
                // Try all 9 combinations of moves for both robots
                for d1 := -1; d1 <= 1; d1++ {
                    nc1 := c1 + d1
                    if nc1 < 0 || nc1 >= cols { continue }
                    for d2 := -1; d2 <= 1; d2++ {
                        nc2 := c2 + d2
                        if nc2 < 0 || nc2 >= cols { continue }
                        // Collect cherries; don't double-count if robots overlap
                        cherries := grid[r][nc1]
                        if nc1 != nc2 { cherries += grid[r][nc2] }
                        if v := dp[c1][c2] + cherries; v > ndp[nc1][nc2] { ndp[nc1][nc2] = v }
                    }
                }
            }
        }
        dp = ndp
    }
    ans := 0
    for _, row := range dp { for _, v := range row { if v > ans { ans = v } } }
    return ans
}

### Rust

In [ ]:
impl Solution {
    pub fn cherry_pickup(grid: Vec<Vec<i32>>) -> i32 {
        let (rows, cols) = (grid.len(), grid[0].len());
        let neg = i32::MIN / 2;
        // dp[c1][c2] = max cherries with robot1 at col c1, robot2 at col c2
        let mut dp = vec![vec![neg; cols]; cols];
        dp[0][cols - 1] = grid[0][0] + if cols > 1 { grid[0][cols - 1] } else { 0 };

        for r in 1..rows {
            let mut ndp = vec![vec![neg; cols]; cols];
            for c1 in 0..cols {
                for c2 in 0..cols {
                    if dp[c1][c2] == neg { continue; }
                    // Try all 9 combinations of moves for both robots
                    for d1 in -1i32..=1 {
                        let nc1 = c1 as i32 + d1;
                        if nc1 < 0 || nc1 >= cols as i32 { continue; }
                        let nc1 = nc1 as usize;
                        for d2 in -1i32..=1 {
                            let nc2 = c2 as i32 + d2;
                            if nc2 < 0 || nc2 >= cols as i32 { continue; }
                            let nc2 = nc2 as usize;
                            // Collect cherries; don't double-count if robots overlap
                            let cherries = grid[r][nc1] + if nc1 != nc2 { grid[r][nc2] } else { 0 };
                            let val = dp[c1][c2] + cherries;
                            if val > ndp[nc1][nc2] { ndp[nc1][nc2] = val; }
                        }
                    }
                }
            }
            dp = ndp;
        }
        dp.iter().flatten().copied().filter(|&v| v != neg).max().unwrap_or(0)
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `grid = [[3,1,1],[2,5,1],[1,5,5],[2,1,1]]`
Robot 1 path (0,0)→(1,0)→(2,1)→(3,0): collects 3+2+5+2=12. Robot 2 path (0,2)→(1,1)→(2,2)→(3,2): collects 1+5+5+1=12. No overlaps. Total: **24**.

### 2. Slightly Complex
**Input:** `grid = [[1,0,0,0,0,0,1],[2,0,0,0,0,3,0],[2,0,9,0,0,0,0],[0,3,0,5,4,0,0],[1,0,2,3,0,0,6]]`
Optimal routing navigates robot 1 toward the 9 and robot 2 toward the rightward bonuses. Answer: **28**.

### 3. Edge Case: Time Factor
**Input:** 70×70 grid.
The DP processes $70 \times 70^2 \times 9 = 3{,}087{,}000$ transitions — maximum work for this problem's constraints.

### 4. Edge Case: Space Factor
**Input:** 70×70 grid.
Only the current and next rows' $70 \times 70 = 4{,}900$ cells are kept in memory — $O(cols^2)$ space with the rolling-array optimisation.

### 5. Almost-Impossible but Plausible
**Input:** 2×70 grid, all zeros except grid[1][35]=100.
Both robots must converge on column 35 in row 1. The DP correctly allows both to occupy (1,35) and counts 100 only once, giving answer **100**.